In [1]:
import numpy as np
import time
import h5py
import epics
from matplotlib import pyplot as plt
import os

In [ ]:
PV_DVF1_NY = "CAX:A:BASLER01:image1:ArraySize0_RBV"
PV_DVF1_NX = "CAX:A:BASLER01:image1:ArraySize1_RBV"
PV_DVF1_IMG = "CAX:A:BASLER01:image1:ArrayData"

PV_DVF2_NY = "CAX:B:BASLER01:image1:ArraySize0_RBV"
PV_DVF2_NX = "CAX:B:BASLER01:image1:ArraySize1_RBV"
PV_DVF2_IMG = "CAX:B:BASLER01:image1:ArrayData"

PV_SLIT1_TOP_SP = "CAX:A:PP02:A.VAL"
PV_SLIT1_BOTTOM_SP = "CAX:A:PP02:B.VAL"
PV_SLIT1_LEFT_SP = "CAX:A:PP02:C.VAL"
PV_SLIT1_RIGHT_SP = "CAX:A:PP02:D.VAL"

PV_SLIT1_TOP_RBV = "CAX:A:PP02:A.RBV"
PV_SLIT1_BOTTOM_RBV = "CAX:A:PP02:B.RBV"
PV_SLIT1_LEFT_RBV = "CAX:A:PP02:C.RBV"
PV_SLIT1_RIGHT_RBV = "CAX:A:PP02:D.RBV"

THRESHOLD = 0.01
TRIALS = 3
DELAY = 4
STEP = 0.1
COUNT_LIM = 5

In [3]:
# niter = 5
# delta = 0.5
# threshold = 0.1

# value = epics.caget(pv_rbv)
# new_value = value

# for i in range(niter*2):
#     print('\n\nIteration: ({:}/{:})'.format(i+1, niter*2))

#     delta *= -1
#     new_value += delta

#     epics.caput(pv_sp, new_value)

#     current_value = epics.caget(pv_rbv)
#     while abs(current_value - new_value) > threshold:
#         time.sleep(4)
#         current_value = epics.caget(pv_rbv)
#         print('Slit is Moving {:.2f}'.format(current_value), end='\r')

# print('Finished')

In [ ]:
def initialize_hdf5(fname, dv: dict):
    
    with h5py.File(fname, 'w') as f:
        f.attrs['begin time'] = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
        keys = list(dv.keys())
        for key in keys:
            f.attrs[key] = dv[key]

def append_dataset_hdf5(fname, img1_array, img2_array, tag, t0, positions):
    
    with h5py.File(fname, 'a') as f:
        dset = f.create_dataset('STEP_{0:03d}_1'.format(tag), data=img1_array, compression="gzip")
        dset.attrs['slit_top'] = positions[0]
        dset.attrs['slit_bottom'] = positions[1]
        dset.attrs['slit_right'] = positions[2]
        dset.attrs['slit_left'] = positions[3]
        dset.attrs['ellapsed time (s)'] = round(time.time() - t0, 3)

        dset2 = f.create_dataset('STEP_{0:03d}_2'.format(tag), data=img2_array, compression="gzip")
        dset2.attrs['slit_top'] = positions[0]
        dset2.attrs['slit_bottom'] = positions[1]
        dset2.attrs['slit_right'] = positions[2]
        dset2.attrs['slit_left'] = positions[3]
        dset2.attrs['ellapsed time (s)'] = round(time.time() - t0, 3)


def move_slit(pv_sp, setpoint, pv_rbv, threshold=THRESHOLD, max_count=COUNT_LIM, delay=DELAY):

    try:
        epics.caput(pv_sp, setpoint)
    except Exception as err:
        raise IOError(f' PUT error: pv, pos = ({epics.caget(pv_rbv)} \n {err}') 
    
    # Check for acknowledgement. Avoid endless loop if command is not properly received.
    icount = 0
    current_value = epics.caget(pv_rbv)
    while abs(current_value - setpoint) > threshold:
        time.sleep(delay)
        current_value = epics.caget(pv_rbv)
        print('Dif: {:} \t New pos: {:} \t Curr: {:}'.format(abs(current_value - setpoint), setpoint, current_value))
        print('Slit is Moving {:.2f}'.format(current_value), end='\r')

        if icount >= max_count:
            print(f'\n WARNING: maximum number of trials to move {pv_sp} reached.'
                    f'\n Current position: {current_value}')
            return False

        icount += 1

    return True

def move_robust_slit(pv_sp, setpoint, pv_rbv, threshold=THRESHOLD, max_count=COUNT_LIM, delay=DELAY, trials=TRIALS):
    ctrials = 0
    status = False
    
    try:
        while ctrials < trials and not status:
            status = move_slit(pv_sp, setpoint, pv_rbv, threshold=THRESHOLD, max_count=COUNT_LIM, delay=DELAY)

            ctrials += 1
        current_value = epics.caget(pv_rbv)
        if not status:
            raise Exception('WARNING: maximum number of trials to move{pv_sp} reached.'
                    f'\n Current position: {current_value}')
        return True
    except Exception:
        print('Not moved')
        return False


# HDF5 initialize

In [ ]:
fname = 'dv1_images.h5'

dv = dict()

dv['slit_top'] = epics.caget(PV_SLIT1_TOP_RBV)
dv['slit_bottom'] = epics.caget(PV_SLIT1_BOTTOM_RBV)
dv['slit_left'] = epics.caget(PV_SLIT1_LEFT_RBV)
dv['slit_right'] = epics.caget(PV_SLIT1_RIGHT_RBV)

dv['STEP'] = 0.1

initialize_hdf5(fname, dv)

# Scan

### Initial Conditions

In [ ]:
slit_positions_top = np.arange(14.81, 19.71+STEP, STEP) #[-3:]
slit_positions_bottom = np.arange(30.9, 35.8+STEP, STEP) #[-3:]
slit_positions_right = np.arange(45.8, 47.2+STEP, STEP) #[-3:]
slit_positions_left = np.arange(42.3, 43.6+STEP, STEP) #[-3:]

dvf1_nx = epics.caget(PV_DVF1_NX)
dvf1_ny = epics.caget(PV_DVF1_NY)


dvf2_nx = epics.caget(PV_DVF2_NX)
dvf2_ny = epics.caget(PV_DVF2_NY)
slit_positions_top

array([14.81, 14.91, 15.01, 15.11, 15.21, 15.31, 15.41, 15.51, 15.61,
       15.71, 15.81, 15.91, 16.01, 16.11, 16.21, 16.31, 16.41, 16.51,
       16.61, 16.71, 16.81, 16.91, 17.01, 17.11, 17.21, 17.31, 17.41,
       17.51, 17.61, 17.71, 17.81, 17.91, 18.01, 18.11, 18.21, 18.31,
       18.41, 18.51, 18.61, 18.71, 18.81, 18.91, 19.01, 19.11, 19.21,
       19.31, 19.41, 19.51, 19.61, 19.71, 19.81])

### First test

In [33]:

# pv_sp = PV_SLIT1_RIGHT_SP
# pv_rbv = PV_SLIT1_RIGHT_RBVicount = 0

# t0 = time.time()
# for i, pos in enumerate(slit_positions_right):

#     positions = [
#         epics.caget(PV_SLIT1_TOP_RBV),
#         epics.caget(PV_SLIT1_BOTTOM_RBV),
#         epics.caget(PV_SLIT1_RIGHT_RBV),
#         epics.caget(PV_SLIT1_LEFT_RBV),
#     ]

#     epics.caput(pv_sp, pos)
    
#     current_value = epics.caget(pv_rbv)
#     while abs(current_value - pos) > threshold:
#         time.sleep(4)
#         current_value = epics.caget(pv_rbv)
#         print('Slit is Moving {:.2f}'.format(current_value), end='\r')

#     time.sleep(0.5)    
#     print('Finished movement {0}/{1}: pos={2:.1f} mm'.format(i+1, len(slit_positions_right), current_value), end='\r')

#     img = np.array(epics.caget(PV_DVF1_IMG))
#     img = img.reshape((dvf1_nx, dvf1_ny))
    
#     append_dataset_hdf5(fname, img, i, t0, positions)

# epics.caput(PV_SLIT1_TOP_SP, slit_positions_top[-1])
# epics.caput(PV_SLIT1_BOTTOM_SP, slit_positions_bottom[-1])
# epics.caput(PV_SLIT1_RIGHT_SP, slit_positions_right[-1])
# epics.caput(PV_SLIT1_LEFT_SP, slit_positions_left[-1])

# print('Finished scan!')

In [ ]:
pvs_sp = [
    PV_SLIT1_TOP_SP,
    PV_SLIT1_BOTTOM_SP,
    PV_SLIT1_RIGHT_SP,
    PV_SLIT1_LEFT_SP,
]
pvs_rbv = [
    PV_SLIT1_TOP_RBV,
    PV_SLIT1_BOTTOM_RBV,
    PV_SLIT1_RIGHT_RBV,
    PV_SLIT1_LEFT_RBV,
]

slit_pos = [
    slit_positions_top,
    slit_positions_bottom,
    slit_positions_right,
    slit_positions_left,
]

epics.caput(PV_SLIT1_TOP_SP, 19.71)
epics.caput(PV_SLIT1_BOTTOM_SP, 35.8)
epics.caput(PV_SLIT1_RIGHT_SP, 47.2)
epics.caput(PV_SLIT1_LEFT_SP, 43.6)

for j, pv_sp in enumerate(pvs_sp):
    pv_rbv = pvs_rbv[j]

    t0 = time.time()
    for i, pos in enumerate(slit_pos[j]):

        positions = [
            epics.caget(PV_SLIT1_TOP_RBV),
            epics.caget(PV_SLIT1_BOTTOM_RBV),
            epics.caget(PV_SLIT1_RIGHT_RBV),
            epics.caget(PV_SLIT1_LEFT_RBV),
        ]

        try:
            epics.caput(pv_sp, pos)
        except Exception as err:
            raise IOError(f' PUT error: pv, pos = ({pv_sp}, {pos:.2f}) \n {err}') 
        
        current_value = epics.caget(pv_rbv)
        
        # Check for acknowledgement. Avoid endless loop if command is not properly received.
        icount, docontinue = 0, False
        while abs(current_value - pos) > THRESHOLD:
            time.sleep(4)
            current_value = epics.caget(pv_rbv)
            print('Dif: {:} \t New pos: {:} \t Curr: {:}'.format(abs(current_value - pos), pos, current_value))
            print('Slit is Moving {:.2f}'.format(current_value), end='\r')

            if icount >= COUNT_LIM:
                print(f'\n WARNING: maximum number of trials to move {pv_sp} reached.'
                      f'\n Current position: {current_value}')
                docontinue = True
                break

            icount += 1

        if docontinue: continue

        time.sleep(0.5)    
        print('\nFinished movement {0}/{1}: pos={2:.1f} mm\n\n'.format(i+1, len(slit_pos[j]), current_value))

        img = np.array(epics.caget(PV_DVF1_IMG))
        img = img.reshape((dvf1_nx, dvf1_ny))
        
        img2 = np.array(epics.caget(PV_DVF2_IMG))
        img2 = img2.reshape((dvf2_nx, dvf2_ny))
        
        append_dataset_hdf5(fname, img, img2, i+100*j, t0, positions)


    epics.caput(PV_SLIT1_TOP_SP, 19.71)
    epics.caput(PV_SLIT1_BOTTOM_SP, 35.8)
    epics.caput(PV_SLIT1_RIGHT_SP, 47.2)
    epics.caput(PV_SLIT1_LEFT_SP, 43.6)

    print('Finished {:} slit scan'.format(['top', 'bottom', 'right', 'left'][j]))

epics.caput(PV_SLIT1_TOP_SP, 18.21)
epics.caput(PV_SLIT1_BOTTOM_SP, 35.3)
epics.caput(PV_SLIT1_RIGHT_SP, 47.8)
epics.caput(PV_SLIT1_LEFT_SP, 43.1)

Dif: 0.0 	 New pos: 14.81 	 Curr: 14.81
Slit is Moving 14.81
Finished movement 1/51: pos=14.8 mm


Dif: 0.0 	 New pos: 14.91 	 Curr: 14.91
Slit is Moving 14.91
Finished movement 2/51: pos=14.9 mm


Dif: 0.0 	 New pos: 15.01 	 Curr: 15.01
Slit is Moving 15.01
Finished movement 3/51: pos=15.0 mm


Dif: 0.0 	 New pos: 15.11 	 Curr: 15.11
Slit is Moving 15.11
Finished movement 4/51: pos=15.1 mm


Dif: 1.7763568394002505e-15 	 New pos: 15.209999999999999 	 Curr: 15.21
Slit is Moving 15.21
Finished movement 5/51: pos=15.2 mm


Dif: 1.7763568394002505e-15 	 New pos: 15.309999999999999 	 Curr: 15.31
Slit is Moving 15.31
Finished movement 6/51: pos=15.3 mm


Dif: 1.7763568394002505e-15 	 New pos: 15.409999999999998 	 Curr: 15.41
Slit is Moving 15.41
Finished movement 7/51: pos=15.4 mm


Dif: 1.7763568394002505e-15 	 New pos: 15.509999999999998 	 Curr: 15.51
Slit is Moving 15.51
Finished movement 8/51: pos=15.5 mm


Dif: 1.7763568394002505e-15 	 New pos: 15.609999999999998 	 Curr: 15.61
Slit is 

1

In [67]:
fname = 'square_scan.h5'

dv = dict()

dv['slit_top'] = epics.caget(PV_SLIT1_TOP_RBV)
dv['slit_bottom'] = epics.caget(PV_SLIT1_BOTTOM_RBV)
dv['slit_left'] = epics.caget(PV_SLIT1_LEFT_RBV)
dv['slit_right'] = epics.caget(PV_SLIT1_RIGHT_RBV)

initialize_hdf5(fname, dv)

In [68]:
THRESHOLD = 0.01
TRIALS = 3
DELAY = 4
STEP = 0.1
COUNT_LIM = 5
GAP_HOR = 0.4
GAP_VER = 0.4

xdeltas = np.linspace(0, 1.3, 6)
ydeltas = np.linspace(0, 4.9, 12)

dvf1_nx = epics.caget(PV_DVF1_NX)
dvf1_ny = epics.caget(PV_DVF1_NY)


dvf2_nx = epics.caget(PV_DVF2_NX)
dvf2_ny = epics.caget(PV_DVF2_NY)
slit_positions_top

array([14.81, 14.91, 15.01, 15.11, 15.21, 15.31, 15.41, 15.51, 15.61,
       15.71, 15.81, 15.91, 16.01, 16.11, 16.21, 16.31, 16.41, 16.51,
       16.61, 16.71, 16.81, 16.91, 17.01, 17.11, 17.21, 17.31, 17.41,
       17.51, 17.61, 17.71, 17.81, 17.91, 18.01, 18.11, 18.21, 18.31,
       18.41, 18.51, 18.61, 18.71, 18.81, 18.91, 19.01, 19.11, 19.21,
       19.31, 19.41, 19.51, 19.61, 19.71, 19.81])

In [69]:
slit_top_pos0 = 14.81 + GAP_VER
slit_bottom_pos0 = 35.8
slit_right_pos0 = 45.8 + GAP_HOR
slit_left_pos0 = 43.6

epics.caput(PV_SLIT1_TOP_SP, slit_top_pos0)
epics.caput(PV_SLIT1_BOTTOM_SP, slit_bottom_pos0)
epics.caput(PV_SLIT1_RIGHT_SP, slit_right_pos0)
epics.caput(PV_SLIT1_LEFT_SP, slit_left_pos0)

1

In [ ]:
for j, ydelta in enumerate(ydeltas):
    move_robust_slit(pv_sp=PV_SLIT1_TOP_SP, setpoint=slit_top_pos0+ydelta, pv_rbv=PV_SLIT1_TOP_RBV)
    move_robust_slit(pv_sp=PV_SLIT1_BOTTOM_SP, setpoint=slit_bottom_pos0-ydelta, pv_rbv=PV_SLIT1_BOTTOM_RBV)
    move_robust_slit(pv_sp=PV_SLIT1_LEFT_SP, setpoint=slit_left_pos0, pv_rbv=PV_SLIT1_LEFT_RBV)
    move_robust_slit(pv_sp=PV_SLIT1_RIGHT_SP, setpoint=slit_right_pos0, pv_rbv=PV_SLIT1_RIGHT_RBV)

    for i, xdelta in enumerate(xdeltas):
        move_robust_slit(pv_sp=PV_SLIT1_LEFT_SP, setpoint=slit_left_pos0-xdelta, pv_rbv=PV_SLIT1_LEFT_RBV)
        move_robust_slit(pv_sp=PV_SLIT1_RIGHT_SP, setpoint=slit_right_pos0+xdelta, pv_rbv=PV_SLIT1_RIGHT_RBV)

        slit_pos_top = epics.caget(PV_SLIT1_TOP_SP)
        slit_pos_bottom = epics.caget(PV_SLIT1_BOTTOM_SP)
        slit_pos_right = epics.caget(PV_SLIT1_RIGHT_SP)
        slit_pos_left = epics.caget(PV_SLIT1_LEFT_SP)

        positions = [
            slit_pos_top,
            slit_pos_bottom,
            slit_pos_right,
            slit_pos_left,
        ]

        img = np.array(epics.caget(PV_DVF1_IMG))
        img = img.reshape((dvf1_nx, dvf1_ny))
        
        img2 = np.array(epics.caget(PV_DVF2_IMG))
        img2 = img2.reshape((dvf2_nx, dvf2_ny))
        
        append_dataset_hdf5(fname, img, img2, i+100*j, t0, positions)

epics.caput(PV_SLIT1_TOP_SP, 18.21)
epics.caput(PV_SLIT1_BOTTOM_SP, 35.3)
epics.caput(PV_SLIT1_RIGHT_SP, 47.8)
epics.caput(PV_SLIT1_LEFT_SP, 43.1)

Dif: 0.0 	 New pos: 43.6 	 Curr: 43.6
Dif: 0.0 	 New pos: 43.34 	 Curr: 43.34
Dif: 7.105427357601002e-15 	 New pos: 46.459999999999994 	 Curr: 46.46
Dif: 0.0 	 New pos: 43.08 	 Curr: 43.08
Dif: 0.0 	 New pos: 46.72 	 Curr: 46.72
Dif: 0.0 	 New pos: 42.82 	 Curr: 42.82
Dif: 7.105427357601002e-15 	 New pos: 46.98 	 Curr: 46.980000000000004
Dif: 0.0 	 New pos: 42.56 	 Curr: 42.56
Dif: 7.105427357601002e-15 	 New pos: 47.239999999999995 	 Curr: 47.24
Dif: 0.0 	 New pos: 42.300000000000004 	 Curr: 42.300000000000004
Dif: 7.105427357601002e-15 	 New pos: 47.49999999999999 	 Curr: 47.5
Dif: 1.4204545454532536e-05 	 New pos: 15.655454545454546 	 Curr: 15.65546875
Dif: 1.4204545450979822e-05 	 New pos: 35.35454545454545 	 Curr: 35.35453125
Dif: 0.0 	 New pos: 43.6 	 Curr: 43.6
Dif: 7.105427357601002e-15 	 New pos: 46.199999999999996 	 Curr: 46.2
Dif: 0.0 	 New pos: 43.34 	 Curr: 43.34
Dif: 7.105427357601002e-15 	 New pos: 46.459999999999994 	 Curr: 46.46
Dif: 0.0 	 New pos: 43.08 	 Curr: 43.08


1